#### 1. Librerías.

In [1]:
%run "./librerias/librerias.ipynb"

#### 2. Constantes.

In [2]:
%run "./constantes/constantes.ipynb"

#### 3. Funciones.

In [3]:
%run "./funciones/funciones.ipynb"

#### 4. Lecturas.

In [4]:
#a. Conexión a la BBDD.
conn = sqlite3.connect(path_db)

In [5]:
#b. Leo las tablas de la BBDD como dataframes de pandas.
#i. Veo que tablas hay en la BBDD.
print("Tablas existentes:")
tablas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tablas)
#ii. Leo las tablas.
print("\nLeyendo las tablas...")
try:
    df_libros = pd.read_sql_query("SELECT * FROM libros;", conn)
    df_lectores = pd.read_sql_query("SELECT * FROM lectores;", conn)
    df_interacciones = pd.read_sql_query("SELECT * FROM interacciones;", conn)
except Exception as e:
    print(f"Error al leer las tablas: {e}")
else:
    print("Tablas leídas correctamente.")
#iii. Cierro la conexión a la BBDD.
conn.close()

Tablas existentes:
            name
0         libros
1       lectores
2  interacciones

Leyendo las tablas...
Tablas leídas correctamente.


In [6]:
#c. Ejemplo a entregar, con los legajos a predecir su top 20 de libros a recomendar.
df_entregar = pd.read_csv("./inputs/ejemplo.csv")

#### 5. Calidad de datos.

In [7]:
#a. Cantidad de filas y columnas.
print("La tabla de libros, la cantidad de filas y columnas es: {}".format(df_libros.shape))
print("La tabla de lectores, la cantidad de filas y columnas es: {}".format(df_lectores.shape))
print("La tabla de interacciones, la cantidad de filas y columnas es: {}".format(df_interacciones.shape))

La tabla de libros, la cantidad de filas y columnas es: (128743, 9)
La tabla de lectores, la cantidad de filas y columnas es: (11285, 5)
La tabla de interacciones, la cantidad de filas y columnas es: (461408, 4)


In [8]:
#b. Me fijo si hay registros duplicados.
#i. Libros.
cant_libros_duplicados = df_libros["id_libro"].duplicated().sum()
print("Hay {} libros repetidos.".format(cant_libros_duplicados))
#ii. Usuarios.
cant_lectores_duplicados = df_lectores["id_lector"].duplicated().sum()
print("Hay {} usuarios repetidos.".format(cant_lectores_duplicados))
#iii. Combinaciones id_lector-id_libro en interacciones.
canti_interacciones_duplicados = df_interacciones[["id_libro","id_lector"]].duplicated().sum()
print("Hay {} lectores-libros repetidos.".format(canti_interacciones_duplicados))

Hay 0 libros repetidos.
Hay 0 usuarios repetidos.
Hay 0 lectores-libros repetidos.


In [9]:
#c. Me fijo la cantidad de usuarios y libros con interacciones.
#i. Libros.
cantidad_libros = len(df_libros["id_libro"].unique())
cantidad_libros_opiniones = len(df_interacciones["id_libro"].unique())
print("Hay {} libros totales, el {}% tienen opiniones.".format(cantidad_libros,round(cantidad_libros_opiniones/cantidad_libros*100,2)))
#ii. Lectores.
cantidad_lectores = len(df_lectores["id_lector"].unique())
cantidad_lectores_opiniones = len(df_interacciones["id_lector"].unique())
print("Hay {} lectores totales, el {}% otorgó opiniones.".format(cantidad_lectores,round(cantidad_lectores_opiniones/cantidad_lectores*100,2)))

Hay 128743 libros totales, el 37.39% tienen opiniones.
Hay 11285 lectores totales, el 94.58% otorgó opiniones.


In [10]:
#d. Cantidad y % de nulos por columna de cada tabla.
#i. Libros.
id_libros_unicos_con_opiniones = list(df_interacciones["id_libro"].unique())
libros_con_opiniones = df_libros[df_libros["id_libro"].isin(id_libros_unicos_con_opiniones)]
resumen_nulos_libros_con_interacciones = pd.DataFrame({
    "nulos": libros_con_opiniones.isnull().sum(),
    "pct_nulos": (libros_con_opiniones.isnull().sum() / len(libros_con_opiniones) * 100).round(2)
})

resumen_nulos_libros_con_interacciones

,nulos,pct_nulos
id_libro,0,0.00
titulo,0,0.00
autor,4,0.01
genero,0,0.00
editorial,8,0.02
anio_edicion,0,0.00
isbn,0,0.00
resumen,41,0.09
img_src,0,0.00


In [11]:
#ii. Lectores.
resumen_nulos_lectores = pd.DataFrame({
    "nulos": df_lectores.isnull().sum(),
    "pct_nulos": (df_lectores.isnull().sum() / len(df_lectores) * 100).round(2)
})
resumen_nulos_lectores

,nulos,pct_nulos
id_lector,0,0.0
nombre,0,0.0
genero,0,0.0
vive_en,0,0.0
nacimiento,0,0.0


In [12]:
#iii. Interacciones.
resumen_nulos_interacciones = pd.DataFrame({
    "nulos": df_interacciones.isnull().sum(),
    "pct_nulos": (df_interacciones.isnull().sum() / len(df_interacciones) * 100).round(2)
})
resumen_nulos_interacciones

,nulos,pct_nulos
id_lector,0,0.0
id_libro,0,0.0
fecha,0,0.0
rating,0,0.0


In [13]:
#e. Tipos de datos.
#i. Libros.
#1. Imprimo.
print(df_libros.dtypes)

#2. Convierto anio_edicion
#a. Limpio anio_edicion: extraigo año válido (4 dígitos consecutivos) y paso rotos/ambiguos a null.
df_libros["anio_edicion"] = df_libros["anio_edicion"].astype(str).str.extract(r"(\d{4})")
df_libros["anio_edicion"] = pd.to_numeric(df_libros["anio_edicion"], errors="coerce")
print(f"Nulos tras extraer año válido: {df_libros['anio_edicion'].isnull().sum()}")

#b. Nulifico años fuera de rango razonable (típicamente errores de digitación o placeholders).
mask_fuera_rango = (df_libros["anio_edicion"] < 1800) | (df_libros["anio_edicion"] > anio_actual)
print(f"Años fuera de rango [1800, {anio_actual}] nulificados: {mask_fuera_rango.sum()}")
df_libros.loc[mask_fuera_rango, "anio_edicion"] = np.nan

#c. Imputo nulos con el promedio de anio_edicion del mismo título.
promedio_por_titulo = df_libros.groupby("titulo")["anio_edicion"].transform("mean").round()
df_libros["anio_edicion"] = df_libros["anio_edicion"].fillna(promedio_por_titulo)
print(f"Nulos tras imputar por promedio de título: {df_libros['anio_edicion'].isnull().sum()}")

#d. Si quedan nulos residuales (título con TODAS las ediciones rotas), imputo con el promedio de la editorial.
promedio_por_editorial = df_libros.groupby("editorial")["anio_edicion"].transform("mean").round()
df_libros["anio_edicion"] = df_libros["anio_edicion"].fillna(promedio_por_editorial)
print(f"Nulos tras imputar por promedio de editorial: {df_libros['anio_edicion'].isnull().sum()}")

#e. Si siguen habiendo nulos residuales (título Y editorial sin ningún año válido), tomo la mediana general de todos los libros.
df_libros["anio_edicion"] = df_libros["anio_edicion"].fillna(round(df_libros["anio_edicion"].median()))
print(f"Nulos tras imputar por mediana general: {df_libros['anio_edicion'].isnull().sum()}")

#f. Ahora si, casteo a int (ya no deberia haber nulos).
df_libros["anio_edicion"] = df_libros["anio_edicion"].astype(int)

id_libro        object
titulo          object
autor           object
genero          object
editorial       object
anio_edicion    object
isbn            object
resumen         object
img_src         object
dtype: object
Nulos tras extraer año válido: 78305
Años fuera de rango [1800, 2024] nulificados: 717
Nulos tras imputar por promedio de título: 78969
Nulos tras imputar por promedio de editorial: 78275
Nulos tras imputar por mediana general: 0


In [14]:
#ii. Lectores.
#1. Imprimo.
print(df_lectores.dtypes)

#2. Convierto Nacimiento.
#a. Limpio nacimiento: fuerzo a numerico, lo que no matchee queda null.
df_lectores["nacimiento"] = pd.to_numeric(df_lectores["nacimiento"], errors="coerce")
print(f"Nulos tras forzar a numérico: {df_lectores['nacimiento'].isnull().sum()}")

#b. Nulifico nacimientos fuera de rango razonable (errores de digitación o placeholders).
mask_fuera_rango = (df_lectores["nacimiento"] < 1900) | (df_lectores["nacimiento"] > anio_actual)
print(f"Nacimientos fuera de rango [1900, {anio_actual}] nulificados: {mask_fuera_rango.sum()}")
df_lectores.loc[mask_fuera_rango, "nacimiento"] = np.nan

#c. Imputo nulos con el promedio de nacimiento del mismo nombre.
promedio_por_nombre = df_lectores.groupby("nombre")["nacimiento"].transform("mean").round()
df_lectores["nacimiento"] = df_lectores["nacimiento"].fillna(promedio_por_nombre)
print(f"Nulos tras imputar por promedio de nombre: {df_lectores['nacimiento'].isnull().sum()}")

#d. Si quedan nulos residuales (nombres donde TODOS los registros estaban rotos), imputo con el promedio global.
df_lectores["nacimiento"] = df_lectores["nacimiento"].fillna(round(df_lectores["nacimiento"].mean()))
print(f"Nulos tras imputar por promedio global: {df_lectores['nacimiento'].isnull().sum()}")

#e. Chequeo que no queden nulos antes de castear.
print("Nulos restantes:", df_lectores["nacimiento"].isnull().sum())

#f. Casteo a int.
df_lectores["nacimiento"] = df_lectores["nacimiento"].astype(int)

id_lector     object
nombre        object
genero        object
vive_en       object
nacimiento    object
dtype: object
Nulos tras forzar a numérico: 3429
Nacimientos fuera de rango [1900, 2024] nulificados: 0
Nulos tras imputar por promedio de nombre: 3167
Nulos tras imputar por promedio global: 0
Nulos restantes: 0


In [15]:
#iii. Interacciones.
#1. Imprimo.
print(df_interacciones.dtypes)

#2. Corrijo Fecha.
#a. Convierto a datetime, lo que no matchee queda NaT.
fecha_original_nulos = df_interacciones["fecha"].isnull().sum()
df_interacciones["fecha"] = pd.to_datetime(df_interacciones["fecha"], format="%d-%m-%Y", errors="coerce")
print(f"Fechas no parseables (NaT) tras convertir: {df_interacciones['fecha'].isnull().sum() - fecha_original_nulos}")

#b. Saco las filas con fecha no parseable (no tiene sentido imputar una fecha de interacción).
filas_antes = len(df_interacciones)
df_interacciones = df_interacciones.dropna(subset=["fecha"])
print(f"Filas eliminadas por fecha inválida: {filas_antes - len(df_interacciones)}")

id_lector    object
id_libro     object
fecha        object
rating        int64
dtype: object
Fechas no parseables (NaT) tras convertir: 1
Filas eliminadas por fecha inválida: 1


In [16]:
#f. Dataset a predecir.
#i. Cantidad de lectores.
cant_lectores_predecir = len(df_entregar["id_lector"].unique())
print("Hay {} lectores únicos a predecir".format(cant_lectores_predecir))

#ii. Cantidad de lectores a predecir que no están en df_lectores (lectores "nuevos", sin atributos conocidos).
lectores_predecir = df_entregar["id_lector"].unique()
lectores_conocidos = df_lectores["id_lector"].unique()

lectores_sin_datos = set(lectores_predecir) - set(lectores_conocidos)
print(f"Hay {len(lectores_sin_datos)} lectores a predecir que NO están en df_lectores.")

#iii. Cantidad de lectores a predecir que si están en df_lectores (para tener el panorama completo).
lectores_con_datos = set(lectores_predecir) & set(lectores_conocidos)
print(f"Hay {len(lectores_con_datos)} lectores a predecir que SÍ están en df_lectores.")

#iv. De los lectores que si están en df_lectores, reviso nulos por columna (¿tienen datos completos o les faltan atributos?).
df_lectores_con_datos = df_lectores[df_lectores["id_lector"].isin(lectores_con_datos)]

resumen_nulos_lectores_predecir = pd.DataFrame({
    "nulos": df_lectores_con_datos.isnull().sum(),
    "pct_nulos": (df_lectores_con_datos.isnull().sum() / len(df_lectores_con_datos) * 100).round(2)
})
print(f"Revisando nulos sobre {len(df_lectores_con_datos)} lectores (los que están en df_lectores y hay que predecirles).")
display(resumen_nulos_lectores_predecir)

#v. Cantidad de lectores a predecir con interacciones (o sea, que tienen historial en df_interacciones).
mask_interacciones_predecir = df_interacciones["id_lector"].isin(lectores_predecir)
lectores_predecir_con_interacciones = df_interacciones.loc[mask_interacciones_predecir, "id_lector"].unique()

print(f"Lectores a predecir con al menos 1 interacción: {len(lectores_predecir_con_interacciones)} de {len(lectores_predecir)}")

#vi. Cantidad de interacciones promedio por lector a predecir.
interacciones_por_lector_predecir = df_interacciones[mask_interacciones_predecir].groupby("id_lector").size()

print(f"Promedio de interacciones por lector a predecir: {interacciones_por_lector_predecir.mean():.2f}")
print(f"Mediana: {interacciones_por_lector_predecir.median():.0f}")
print(f"Mínimo: {interacciones_por_lector_predecir.min()}")
print(f"Máximo: {interacciones_por_lector_predecir.max()}")


Hay 832 lectores únicos a predecir
Hay 1 lectores a predecir que NO están en df_lectores.
Hay 831 lectores a predecir que SÍ están en df_lectores.
Revisando nulos sobre 831 lectores (los que están en df_lectores y hay que predecirles).


,nulos,pct_nulos
id_lector,0,0.0
nombre,0,0.0
genero,0,0.0
vive_en,0,0.0
nacimiento,0,0.0


Lectores a predecir con al menos 1 interacción: 816 de 832
Promedio de interacciones por lector a predecir: 162.22
Mediana: 95
Mínimo: 1
Máximo: 2260


#### 6. Análisis Exploratorio de Datos (EDA) y Correcciones.

In [17]:
#a. Libros.
#i. Exploración.
#1. Autor: distribución y pareto (tabla).
dist_autor = df_libros["autor"].value_counts().reset_index()
dist_autor.columns = ["autor", "cantidad"]
dist_autor["pct"] = (dist_autor["cantidad"] / dist_autor["cantidad"].sum() * 100).round(2)
dist_autor["pct_acumulado"] = dist_autor["pct"].cumsum().round(2)

cant_autores_total = dist_autor["autor"].nunique()
cant_autores_80pct = (dist_autor["pct_acumulado"] <= 80).sum() + 1  # +1 para incluir el autor que cruza el 80%

print(f"Cantidad total de autores: {cant_autores_total}")
print(f"Cantidad de autores que explican el 80% de los libros: {cant_autores_80pct} ({(cant_autores_80pct / cant_autores_total * 100):.2f}% del total de autores)")
print("Pareto de autores (top 15):")
display(dist_autor.head(15))

#2. Género: distribución, pareto (tabla) y gráfico.
dist_genero = df_libros["genero"].value_counts().reset_index()
dist_genero.columns = ["genero", "cantidad"]
dist_genero["pct"] = (dist_genero["cantidad"] / dist_genero["cantidad"].sum() * 100).round(2)
dist_genero["pct_acumulado"] = dist_genero["pct"].cumsum().round(2)

cant_generos_total = dist_genero["genero"].nunique()
cant_generos_80pct = (dist_genero["pct_acumulado"] <= 80).sum() + 1

print(f"Cantidad total de géneros: {cant_generos_total}")
print(f"Cantidad de géneros que explican el 80% de los libros: {cant_generos_80pct} ({(cant_generos_80pct / cant_generos_total * 100):.2f}% del total de géneros)")
print("Pareto de géneros (top 15):")
display(dist_genero.head(15))

#3. Editorial: distribución y pareto (tabla).
dist_editorial = df_libros["editorial"].value_counts().reset_index()
dist_editorial.columns = ["editorial", "cantidad"]
dist_editorial["pct"] = (dist_editorial["cantidad"] / dist_editorial["cantidad"].sum() * 100).round(2)
dist_editorial["pct_acumulado"] = dist_editorial["pct"].cumsum().round(2)

cant_editoriales_total = dist_editorial["editorial"].nunique()
cant_editoriales_80pct = (dist_editorial["pct_acumulado"] <= 80).sum() + 1

print(f"Cantidad total de editoriales: {cant_editoriales_total}")
print(f"Cantidad de editoriales que explican el 80% de los libros: {cant_editoriales_80pct} ({(cant_editoriales_80pct / cant_editoriales_total * 100):.2f}% del total de editoriales)")
print("Pareto de editoriales (top 15):")
display(dist_editorial.head(15))

#4. Año de edición: distribución, pareto (tabla) y gráfico.
anio_edicion_num = pd.to_numeric(df_libros["anio_edicion"], errors="coerce")

dist_anio = anio_edicion_num.value_counts().reset_index()
dist_anio.columns = ["anio_edicion", "cantidad"]
dist_anio = dist_anio.sort_values("cantidad", ascending=False).reset_index(drop=True)
dist_anio["pct"] = (dist_anio["cantidad"] / dist_anio["cantidad"].sum() * 100).round(2)
dist_anio["pct_acumulado"] = dist_anio["pct"].cumsum().round(2)

cant_anios_total = dist_anio["anio_edicion"].nunique()
cant_anios_80pct = (dist_anio["pct_acumulado"] <= 80).sum() + 1

print(f"Cantidad total de años de edición distintos: {cant_anios_total}")
print(f"Cantidad de años que explican el 80% de los libros: {cant_anios_80pct} ({(cant_anios_80pct / cant_anios_total * 100):.2f}% del total de años)")
print("Pareto de años de edición (top 15):")
display(dist_anio.head(15))

#5. Año de edición: detección de valores raros/atípicos.
#i. Rango general y estadísticas descriptivas.
print("Estadísticas de anio_edicion (numérico):")
display(anio_edicion_num.describe())

#ii. Resumen de rangos (bins por siglo/período) para dimensionar cada bloque.
bins = [-float("inf"), 0, 1800, 1900, 1950, 2000, anio_actual, float("inf")]
labels = ["<=0", "1-1799", "1800-1899", "1900-1949", "1950-1999", "2000-actual", "post-actual"]
dist_rangos = pd.cut(anio_edicion_num, bins=bins, labels=labels).value_counts().reindex(labels)
print("\nCantidad de libros por rango de año:")
display(dist_rangos)

#iii. Libros con año == 0 o negativo (posibles placeholders de "sin dato").
libros_placeholder = df_libros[anio_edicion_num <= 0]
print(f"\nLibros con año <= 0 (posible placeholder): {len(libros_placeholder)}")
display(libros_placeholder[["id_libro", "titulo", "autor", "anio_edicion"]].head(15))

#6. Resumen: longitud promedio, mínima y máxima (tabla resumen).
longitud_resumen = df_libros["resumen"].dropna().str.len()

resumen_longitud = pd.DataFrame({
    "métrica": ["longitud_promedio", "longitud_minima", "longitud_maxima"],
    "valor": [longitud_resumen.mean().round(2), longitud_resumen.min(), longitud_resumen.max()]
})
print("Longitud de resúmenes (en caracteres):")
display(resumen_longitud)

Cantidad total de autores: 21655
Cantidad de autores que explican el 80% de los libros: 21656 (100.00% del total de autores)
Pareto de autores (top 15):


,autor,cantidad,pct,pct_acumulado
0,VV.AA.,475,0.94,0.94
1,"SIERRA I FABRA, JORDI",121,0.24,1.18
2,"KING, STEPHEN",111,0.22,1.40
3,"ROBERTS, NORA",109,0.22,1.62
4,"ASIMOV, ISAAC",103,0.20,1.82
5,"CHRISTIE, AGATHA",96,0.19,2.01
6,"PÉREZ GALDÓS, BENITO",93,0.18,2.19
7,"VÁZQUEZ-FIGUEROA, ALBERTO",78,0.15,2.34
8,"VIDAL, CÉSAR",76,0.15,2.49
9,"CAMILLERI, ANDREA",76,0.15,2.64


Cantidad total de géneros: 66
Cantidad de géneros que explican el 80% de los libros: 10 (15.15% del total de géneros)
Pareto de géneros (top 15):


,genero,cantidad,pct,pct_acumulado
0,Narrativa,8638,17.10,17.10
1,"Novela negra, intriga, terror",6874,13.61,30.71
2,Ensayo,4472,8.85,39.56
3,Infantil y juvenil,3973,7.86,47.42
4,Histórica y aventuras,3876,7.67,55.09
5,"Fantástica, ciencia ficción",2999,5.94,61.03
6,"Romántica, erótica",2953,5.84,66.87
7,Ficción literaria,2750,5.44,72.31
8,Literatura contemporánea,2601,5.15,77.46
9,"Cómics, Novela Gráfica",2197,4.35,81.81


Cantidad total de editoriales: 2909
Cantidad de editoriales que explican el 80% de los libros: 159 (5.47% del total de editoriales)
Pareto de editoriales (top 15):


...

anio_edicion
<=0                 0
1-1799              0
1800-1899          12
1900-1949          53
1950-1999        5281
2000-actual    123397
post-actual         0
Name: count, dtype: int64


Libros con año <= 0 (posible placeholder): 0


,id_libro,titulo,autor,anio_edicion


Longitud de resúmenes (en caracteres):


,métrica,valor
0,longitud_promedio,723.49
1,longitud_minima,0.00
2,longitud_maxima,4523.00


In [18]:
#ii. Corrijo.
#1. Limpieza general de las columnas de tipo texto ---> Paso los valores a minúsculas y saco tildes.
for col in ["titulo", "autor", "genero", "editorial", "resumen"]:
    df_libros[col] = df_libros[col].str.lower().apply(quitar_tildes)

#2. Lleno los vacíos de "resumen" con guiones (placeholder explícito, no NaN silencioso).
df_libros["resumen"] = df_libros["resumen"].fillna("-")

#3. Agrupo genero en categorías más amplias, pensando en un sistema de recomendación.
#a. Agrupo.
df_libros["genero_agrupado"] = df_libros["genero"].apply(mapear_genero_libro)
#b. Reviso: ¿queda algo sin mapear?
sin_mapear = set(df_libros["genero"].unique()) - set(v for vals in map_genero.values() for v in vals)
print(f"Categorías sin mapear: {sin_mapear}")

print(f"\nCantidad de categorías tras agrupar: {df_libros['genero_agrupado'].nunique()}")
display(df_libros["genero_agrupado"].value_counts())

#4. Agrupo editorial: top-N + "otras", conservando frecuencia para la cola larga.
#a. Defino el top-N en base al pareto (ej. top 30, que cubre bastante más del 50% ya).
top_n_editoriales = df_libros["editorial"].value_counts().head(30).index.tolist()

df_libros["editorial_agrupada"] = np.where(
    df_libros["editorial"].isin(top_n_editoriales),
    df_libros["editorial"],
    "otras"
)

print(f"Categorías tras agrupar: {df_libros['editorial_agrupada'].nunique()}")
display(df_libros["editorial_agrupada"].value_counts())

Categorías sin mapear: {None}

Cantidad de categorías tras agrupar: 32


genero_agrupado
narrativa                     8639
terror                        6878
ensayo                        4472
juvenil                       4044
historica_aventuras           3910
ciencia_ficcion_fantastica    3000
romantica_erotica             2957
ficcion_literaria             2750
literatura_contemporanea      2601
comics                        2197
biografias                    2089
lecturas_complementarias      2009
poesia_teatro                 1234
varios                        1023
no_ficcion                     994
clasicos                       756
humor                          458
historia                       188
autoayuda_espiritualidad       131
economia_empresa                50
arte_musica                     47
deportes_juegos                 21
pedagogia                       21
cocina                          19
salud_medicina                  16
sociedad_filosofia               7
audiovisual                      3
viajes                           3
idio

Categorías tras agrupar: 31


editorial_agrupada
otras                      103798
planeta                      1966
ediciones b                  1831
debolsillo                   1750
alfaguara                    1563
anagrama                     1490
alianza                      1324
autor-editor                 1210
plaza & janes                1103
destino                      1006
tusquets                      920
espasa                        907
seix barral                   804
salamandra                    723
edhasa                        693
rba                           659
grijalbo                      637
roca                          578
siruela                       564
booket                        549
la esfera de los libros       547
lumen                         509
sm                            501
norma                         423
catedra                       399
martinez roca                 398
timunmas                      396
maeva                         384
punto de lectura             

In [19]:
#b. Lectores.
#i. Exploración.
#1. Género: distribución (tabla y gráfico).
dist_genero_lector = df_lectores["genero"].value_counts(dropna=False).reset_index()
dist_genero_lector.columns = ["genero", "cantidad"]
dist_genero_lector["pct"] = (dist_genero_lector["cantidad"] / dist_genero_lector["cantidad"].sum() * 100).round(2)
print("Distribución de lectores por género:")
display(dist_genero_lector)

#2. Vive_en: distribución y pareto (tabla), sobre el dato crudo.
dist_vive_en = df_lectores["vive_en"].value_counts(dropna=False).reset_index()
dist_vive_en.columns = ["vive_en", "cantidad"]
dist_vive_en["pct"] = (dist_vive_en["cantidad"] / dist_vive_en["cantidad"].sum() * 100).round(2)
dist_vive_en["pct_acumulado"] = dist_vive_en["pct"].cumsum().round(2)

cant_ubicaciones_total = dist_vive_en["vive_en"].nunique()
cant_ubicaciones_80pct = (dist_vive_en["pct_acumulado"] <= 80).sum() + 1

print(f"Cantidad total de ubicaciones distintas: {cant_ubicaciones_total}")
print(f"Cantidad de ubicaciones que explican el 80% de los lectores: {cant_ubicaciones_80pct} ({(cant_ubicaciones_80pct / cant_ubicaciones_total * 100):.2f}% del total de ubicaciones)")
print("Pareto de ubicaciones (top 15):")
display(dist_vive_en.head(15))

#3. Vive_en: separo en pais y ciudad, limpiando placeholders.
#a. Separo por " - ": si hay 2 partes, la primera es ciudad y la segunda país.
vive_en_str = df_lectores["vive_en"].astype(str).str.strip()
vive_en_str = vive_en_str.replace("nan", np.nan)

#i. Caso especial: strings que terminan en "-" (el país quedó vacío tras el strip, ej. "Madrid -").
termina_en_guion = vive_en_str.str.endswith("-", na=False)
vive_en_str = vive_en_str.where(~termina_en_guion, vive_en_str + " ")

partes = vive_en_str.str.split(" - ", n=1, expand=True)

ciudad_raw = np.where(partes[1].notna(), partes[0].str.strip(), np.nan)
pais_raw = np.where(partes[1].notna(), partes[1].str.strip(), partes[0].str.strip())

df_lectores["ciudad"] = pd.Series(ciudad_raw, index=df_lectores.index)
df_lectores["pais"] = pd.Series(pais_raw, index=df_lectores.index)

#b. Limpio placeholders (vacíos, variantes de "¿?") en CADA columna resultante.
patron_placeholder = r"^\s*$|^\W*\?+\W*$"
df_lectores["ciudad"] = df_lectores["ciudad"].replace(patron_placeholder, np.nan, regex=True)
df_lectores["pais"] = df_lectores["pais"].replace(patron_placeholder, np.nan, regex=True)

#c. Normalizo texto: minúsculas y sin tildes, para consolidar duplicados (ej. "Peru" vs "Perú", "Argentina" vs "ARGENTINA").
#   Muevo este paso ACÁ (antes vivía en la celda de corrección) para que la inferencia y el agrupamiento de más abajo trabajen sobre texto ya consolidado.
for col in ["ciudad", "pais"]:
    df_lectores[col] = df_lectores[col].str.lower().apply(quitar_tildes)

#d. Reviso resultado.
print(f"Nulos en pais: {df_lectores['pais'].isnull().sum()}")
print(f"Nulos en ciudad: {df_lectores['ciudad'].isnull().sum()}")
print("\nDistribución de país (top 15):")
display(df_lectores["pais"].value_counts(dropna=False).head(15))
print("\nDistribución de ciudad (top 15):")
display(df_lectores["ciudad"].value_counts(dropna=False).head(15))

#4. Nacimiento: estadísticas descriptivas y detección de valores raros.
nacimiento_num = pd.to_numeric(df_lectores["nacimiento"], errors="coerce")

print("Estadísticas de nacimiento (numérico):")
display(nacimiento_num.describe())

#a. Valores no numéricos.
mask_no_numerico = df_lectores["nacimiento"].notna() & nacimiento_num.isna()
print(f"\nValores no numéricos en nacimiento: {mask_no_numerico.sum()}")

#b. Lectores con nacimiento implausible (muy viejo, ej. antes de 1900) o futuro.
lectores_raros = df_lectores[(nacimiento_num < 1900) | (nacimiento_num > anio_actual)]
print(f"\nLectores con nacimiento < 1900 o > {anio_actual}: {len(lectores_raros)}")
display(lectores_raros[["id_lector", "nombre", "genero", "vive_en", "nacimiento"]].head(15))

#c. Lectores "menores" implausibles (nacimiento muy reciente, edad < 5 años).
lectores_muy_jovenes = df_lectores[nacimiento_num > anio_actual - 5]
print(f"\nLectores con nacimiento posterior a {anio_actual - 5} (edad < 5 años): {len(lectores_muy_jovenes)}")
display(lectores_muy_jovenes[["id_lector", "nombre", "genero", "vive_en", "nacimiento"]].head(15))

#5. Infiero pais faltante a partir de ciudad, usando el país más frecuente para esa ciudad (cuando ambos son conocidos).
mapa_ciudad_pais = (
    df_lectores.dropna(subset=["ciudad", "pais"])
    .groupby("ciudad")["pais"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
)

mask_pais_faltante = df_lectores["pais"].isna() & df_lectores["ciudad"].notna()
print(f"Filas con pais faltante pero ciudad conocida (candidatas a inferir): {mask_pais_faltante.sum()}")

pais_inferido = df_lectores.loc[mask_pais_faltante, "ciudad"].map(mapa_ciudad_pais)
df_lectores.loc[mask_pais_faltante, "pais"] = pais_inferido

print(f"Nulos en pais tras inferir desde ciudad: {df_lectores['pais'].isnull().sum()}")

#6. Agrupo pais: top-N + "otras", igual que hicimos con editorial.
top_n_paises = df_lectores["pais"].value_counts().head(20).index.tolist()
df_lectores["pais_agrupado"] = np.where(
    df_lectores["pais"].isin(top_n_paises),
    df_lectores["pais"],
    "otras"
)

#7. Nombre: reviso si id_lector y nombre son siempre distintos, o si hay casos donde id_lector parece ser el nombre real.
print(f"Cantidad de id_lector que coinciden exactamente con nombre: {(df_lectores['id_lector'] == df_lectores['nombre']).sum()}")

Distribución de lectores por género:


,genero,cantidad,pct
0,Mujer,3925,34.78
1,Hombre,3712,32.89
2,-,3648,32.33


Cantidad total de ubicaciones distintas: 1585
Cantidad de ubicaciones que explican el 80% de los lectores: 129 (8.14% del total de ubicaciones)
Pareto de ubicaciones (top 15):


,vive_en,cantidad,pct,pct_acumulado
0,España,3068,27.19,27.19
1,Madrid - España,969,8.59,35.78
2,,534,4.73,40.51
3,¿?,457,4.05,44.56
4,Barcelona - España,377,3.34,47.90
5,Valencia - España,213,1.89,49.79
6,Sevilla - España,199,1.76,51.55
7,Buenos aires - Argentina,159,1.41,52.96
8,Mexico,155,1.37,54.33
9,Argentina,135,1.20,55.53


Nulos en pais: 1082
Nulos en ciudad: 4546

Distribución de país (top 15):


...


Valores no numéricos en nacimiento: 0

Lectores con nacimiento < 1900 o > 2024: 0


,id_lector,nombre,genero,vive_en,nacimiento



Lectores con nacimiento posterior a 2019 (edad < 5 años): 0


,id_lector,nombre,genero,vive_en,nacimiento


Filas con pais faltante pero ciudad conocida (candidatas a inferir): 91
Nulos en pais tras inferir desde ciudad: 1009
Cantidad de id_lector que coinciden exactamente con nombre: 1029


In [20]:
#ii. Corrijo.
#1. Modifico el género "-" por el género más frecuente para el mismo nombre (si se puede inferir).
#a. Extraigo el primer nombre (primera palabra de "nombre") para buscar coincidencias de género.
#df_lectores["primer_nombre"] = df_lectores["nombre"].astype(str).str.strip().str.split().str[0]
#b. Armo un diccionario primer_nombre -> género más frecuente, usando SOLO registros con género informado.
#moda_genero_por_nombre = (
#    df_lectores[df_lectores["genero"] != "-"]
#    .groupby("primer_nombre")["genero"]
#    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
#)
#print(f"Cantidad de lectores con género '-': {(df_lectores['genero'] == '-').sum()}")
#c. Reemplazo "-" por el género inferido, donde exista match.
#mask_sin_genero = df_lectores["genero"] == "-"
#genero_inferido = df_lectores.loc[mask_sin_genero, "primer_nombre"].map(moda_genero_por_nombre)
#df_lectores.loc[mask_sin_genero, "genero"] = genero_inferido
#d. Los que no se pudieron inferir (nombre no aparece en ningún otro registro con género informado) vuelven a "-".
#df_lectores["genero"] = df_lectores["genero"].fillna("-")
#print(f"Cantidad de lectores con género '-' tras la corrección: {(df_lectores['genero'] == '-').sum()}")
#e. Limpio la columna auxiliar.
#df_lectores = df_lectores.drop(columns=["primer_nombre"])

In [21]:
#c. Interacciones.
#i. Exploración.
#1. Shape y tipos.
print(f"Shape: {df_interacciones.shape}")
print(df_interacciones.dtypes)

#2. Rating: estadísticas descriptivas y distribución.
print("\nEstadísticas de rating:")
display(df_interacciones["rating"].describe())

dist_rating = df_interacciones["rating"].value_counts(dropna=False).sort_index().reset_index()
dist_rating.columns = ["rating", "cantidad"]
dist_rating["pct"] = (dist_rating["cantidad"] / dist_rating["cantidad"].sum() * 100).round(2)
print("\nDistribución de rating:")
display(dist_rating)

#3. Rating: nulos y valores fuera de rango esperado (asumiendo escala 1-10 según la muestra).
print(f"\nNulos en rating: {df_interacciones['rating'].isnull().sum()}")
rating_fuera_rango = df_interacciones[(df_interacciones["rating"] < 1) | (df_interacciones["rating"] > 10)]
print(f"Ratings fuera de rango [1, 10]: {len(rating_fuera_rango)}")
display(rating_fuera_rango.head(15))

#4. Fecha: conversión a datetime y detección de valores no parseables.
fecha_dt = pd.to_datetime(df_interacciones["fecha"], errors="coerce")
mask_fecha_invalida = df_interacciones["fecha"].notna() & fecha_dt.isna()
print(f"\nFechas no parseables: {mask_fecha_invalida.sum()}")
if mask_fecha_invalida.sum() > 0:
    display(df_interacciones.loc[mask_fecha_invalida, "fecha"].value_counts().head(15))

#5. Fecha: rango temporal y valores extremos.
print(f"\nFecha mínima: {fecha_dt.min()}")
print(f"Fecha máxima: {fecha_dt.max()}")
print(f"Nulos en fecha (original): {df_interacciones['fecha'].isnull().sum()}")

limite_fecha = pd.Timestamp(year=anio_actual, month=12, day=31)
fechas_futuras = df_interacciones[fecha_dt > limite_fecha]
print(f"\nInteracciones con fecha futura: {len(fechas_futuras)}")
display(fechas_futuras.head(15))

Shape: (461407, 4)
id_lector            object
id_libro             object
fecha        datetime64[ns]
rating                int64
dtype: object

Estadísticas de rating:


count    461407.000000
mean          7.263399
std           1.824603
min           1.000000
25%           6.000000
50%           7.000000
75%           8.000000
max          10.000000
Name: rating, dtype: float64


Distribución de rating:


,rating,cantidad,pct
0,1,5173,1.12
1,2,5124,1.11
2,3,4567,0.99
3,4,18870,4.09
4,5,25328,5.49
5,6,87527,18.97
6,7,85018,18.43
7,8,122290,26.50
8,9,53567,11.61
9,10,53943,11.69



Nulos en rating: 0
Ratings fuera de rango [1, 10]: 0


,id_lector,id_libro,fecha,rating



Fechas no parseables: 0

Fecha mínima: 2008-02-24 00:00:00
Fecha máxima: 2024-12-31 00:00:00
Nulos en fecha (original): 0

Interacciones con fecha futura: 0


,id_lector,id_libro,fecha,rating


#### 7. Procesamiento.

In [22]:
#a. Mergeo las tablas.
#i. Renombro columnas para que no sea confuso luego del merge su origen.
df_libros.rename({
    "genero":"genero_libro",
    "genero_agrupado":"genero_libro_agrupado"
},axis=1,inplace=True)

df_lectores.rename({
    "genero":"genero_persona",
    "pais":"pais_persona",
    "pais_agrupado":"pais_persona_agrupado",
    "ciudad":"ciudad_persona"
},axis=1,inplace=True)

#ii. Mergeo.
df_total = df_interacciones.merge(df_libros,how="left",on="id_libro").merge(df_lectores,how="left",on="id_lector")

In [23]:
#b. Selecciono columnas de interés.
df_total = df_total[['id_lector', 'id_libro', 'fecha', 'rating',                              # Interacciones.
                    'autor', 'genero_libro_agrupado', 'editorial_agrupada', 'anio_edicion',   # Libros.
                    'genero_persona', 'pais_persona_agrupado', 'nacimiento']]                 # Personas.
# Nota: saco "ciudad_persona" (demasiada cardinalidad para dummies), agrego "autor" (lo necesito para features lector x autor).

In [24]:
#c. Me fijo combinaciones únicas para, también, entender si me conviene sacar promedios, o no.

#i. Lector - Libro.
combos_libro = df_total[["id_lector","id_libro"]].drop_duplicates()
interacciones_por_lector_libro = combos_libro.groupby("id_lector").size()

#ii. Lector - Autor.
combos_autor = df_total[["id_lector","autor"]].drop_duplicates()
autores_por_lector = combos_autor.groupby("id_lector").size()

repeticion_autor = (
    df_total.groupby(["id_lector","autor"]).size().rename("n_interacciones").reset_index()
)
repeticion_autor["repetido"] = repeticion_autor["n_interacciones"] > 1
pct_autores_repetidos_por_lector = repeticion_autor.groupby("id_lector")["repetido"].mean()

#iii. Lector - Género de libro.
combos_genero = df_total[["id_lector","genero_libro_agrupado"]].drop_duplicates()
generos_por_lector = combos_genero.groupby("id_lector").size()

repeticion_genero = (
    df_total.groupby(["id_lector","genero_libro_agrupado"]).size().rename("n_interacciones").reset_index()
)
repeticion_genero["repetido"] = repeticion_genero["n_interacciones"] > 1
pct_generos_repetidos_por_lector = repeticion_genero.groupby("id_lector")["repetido"].mean()

#d. Resumen consolidado: cuántos items distintos, y qué % de ellos repite.
resumen = pd.DataFrame({
    "libro_distintos_leidos_por_lector":  interacciones_por_lector_libro.describe(),
    "autor_distintos_leidos_por_lector":  autores_por_lector.describe(),
    "genero_distintos_leidos_por_lector": generos_por_lector.describe(),
    #"libro_pct_repetido":     pct_libros_repetidos_por_lector.describe(),
    "pct_lector_repite_autor":     pct_autores_repetidos_por_lector.describe(),
    "pct_lector_repite_genero":    pct_generos_repetidos_por_lector.describe(),
}).T

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
resumen[["mean","50%","std","min","max"]].rename(columns={"50%":"mediana"})

,mean,mediana,std,min,max
libro_distintos_leidos_por_lector,43.231,9.000,101.129,1.000,"2,260.000"
autor_distintos_leidos_por_lector,26.037,7.000,55.350,1.000,951.000
genero_distintos_leidos_por_lector,5.706,4.000,4.652,1.000,24.000
pct_lector_repite_autor,0.198,0.167,0.227,0.000,1.000
pct_lector_repite_genero,0.492,0.500,0.354,0.000,1.000


In [25]:
#d. Calculo edad del lector al momento de la interacción (año de la interacción, no de la edición del libro).
df_total["edad_al_interactuar"] = df_total["fecha"].dt.year - df_total["nacimiento"]

In [26]:
#e. Calculo cantidad de dias que pasaron de la interacción, tomando como referencia el 31/12 del año actual.
fecha_referencia = pd.Timestamp(year=anio_actual, month=12, day=31)

df_total["dias_transcurridos_interaccion"] = (fecha_referencia - df_total["fecha"]).dt.days

In [27]:
#f. Calculo años transcurridos desde la edición.
#i. Hasta la interacción.
df_total["anios_transcurridos_edicion"] = df_total["fecha"].dt.year - df_total["anio_edicion"]
#ii. Hasta hoy.
df_total["antiguedad_libro_hoy"] = anio_actual - df_total["anio_edicion"]
#iii. Ya no necesito "fecha".
df_total.drop(columns=["fecha"], inplace=True)

In [28]:
#g. Frecuencia de lectura del lector y del libro (leave-one-out: no cuento el registro actual).
#i. Lector.
df_total["frecuencia_lector"] = df_total.groupby("id_lector")["id_lector"].transform("count") - 1

#ii. Libro. Como id_lector-id_libro es único, esto YA ES la cantidad de lectores distintos del libro
#    (no hace falta una variable de "popularidad" aparte).
df_total["frecuencia_libro"] = df_total.groupby("id_libro")["id_libro"].transform("count") - 1

#iii. Lector x Autor (cuántas veces este lector interactuó con este autor, sin contar la fila actual).
df_total["n_interacciones_lector_autor"] = df_total.groupby(["id_lector","autor"])["autor"].transform("count") - 1

#iv. Lector x Género (idem, para género).
df_total["n_interacciones_lector_genero"] = df_total.groupby(["id_lector","genero_libro_agrupado"])["genero_libro_agrupado"].transform("count") - 1

#v. Popularidad de autor: cuántos lectores DISTINTOS tuvo, sin contar al lector actual.
#    Acá SÍ hace falta el ajuste condicional, porque lector-autor puede repetirse (pct_repite_autor ≈ 20%),
#    a diferencia de lector-libro que es único por construcción.
n_autores_lector_total = df_total.groupby(["id_lector","autor"])["id_lector"].transform("count")
n_lectores_autor_total = df_total.groupby("autor")["id_lector"].transform("nunique")
df_total["n_lectores_distintos_autor"] = n_lectores_autor_total - (n_autores_lector_total == 1).astype(int)
# Si este lector aparece 1 sola vez para este autor -> restarlo 1 vez de la cuenta de únicos.
# Si aparece más de 1 vez -> nunique ya lo contó una sola vez, no hay nada más que restar.

In [29]:
#h. Features de rating promedio (leave-one-out, evita leakage).
#i. ¿Cómo suele rankear el lector?
df_total["rating_prom_id_lector"] = loo_mean(df_total, "id_lector")
df_total["rating_prom_id_lector_autor"] = loo_mean(df_total, ["id_lector","autor"])
df_total["rating_prom_id_lector_genero_libro_agrupado"] = loo_mean(df_total, ["id_lector","genero_libro_agrupado"])

#ii. ¿Cual suele ser el ranking global?
df_total["rating_prom_id_libro"] = loo_mean(df_total, "id_libro")
df_total["rating_prom_autor"] = loo_mean(df_total, "autor")
df_total["rating_prom_genero"] = loo_mean(df_total, "genero_libro_agrupado")

#iii. Fallback jerárquico para los NaN que deja el LOO cuando n=1 en el grupo.
media_global = df_total["rating"].mean()

#1. Primero limpio los niveles "base" (lector, libro, autor, género) con la media global.
df_total.fillna({
    "rating_prom_id_lector": media_global,
    "rating_prom_id_libro": media_global,
    "rating_prom_autor": media_global,
    "rating_prom_genero": media_global,
}, inplace=True)

#2. Recién ahora uso rating_prom_id_lector (ya sin NaN) como fallback para los niveles de a pares.
for col, fallback in [
    ("rating_prom_id_lector_autor", "rating_prom_id_lector"),
    ("rating_prom_id_lector_genero_libro_agrupado", "rating_prom_id_lector"),
]:
    df_total[col] = df_total[col].fillna(df_total[fallback])

In [30]:
#i. Diversidad del lector ---> cuántos autores/géneros distintos leyó.
df_total = df_total.merge(autores_por_lector.rename("n_autores_distintos_lector"), on="id_lector", how="left")
df_total = df_total.merge(generos_por_lector.rename("n_generos_distintos_lector"), on="id_lector", how="left")

In [31]:
#j. Codifico categóricas de baja/media cardinalidad con dummies.
cols_dummies = ["genero_persona"]#,"pais_persona_agrupado","genero_libro_agrupado", "editorial_agrupada"]
df_total = pd.get_dummies(df_total, columns=cols_dummies, drop_first=False)

In [32]:
#k. Encodeo categoricas de alta cardinalidad con frequency encoding (evita explosion de columnas).
#df_total["id_lector_freq"] = df_total["id_lector"].map(df_total["id_lector"].value_counts())
#df_total["id_libro_freq"] = df_total["id_libro"].map(df_total["id_libro"].value_counts())

In [33]:
#l. Forma final.
print(f"Train: {df_total.shape}")

Train: (461407, 29)


In [34]:
#m. Me aseguro que no hayan quedado nulos en los ratings.
cols_rating = [c for c in df_total.columns if c.startswith("rating_prom")]
print(df_total[cols_rating].isna().sum())

rating_prom_id_lector                          0
rating_prom_id_lector_autor                    0
rating_prom_id_lector_genero_libro_agrupado    0
rating_prom_id_libro                           0
rating_prom_autor                              0
rating_prom_genero                             0
dtype: int64


#### 8. Entrenamiento.

In [ ]:
#a. Armo X e y. 
features_base = [
    #"id_lector_freq", "id_libro_freq",
    "anio_edicion", "nacimiento",
    "edad_al_interactuar", "dias_transcurridos_interaccion",
    "anios_transcurridos_edicion", "antiguedad_libro_hoy",
    'frecuencia_lector', 
    'frecuencia_libro', 
    'n_interacciones_lector_autor',
    'n_interacciones_lector_genero', 
    'n_lectores_distintos_autor',
    #'rating_prom_id_lector', 
    'rating_prom_id_lector_autor',
    'rating_prom_id_lector_genero_libro_agrupado', 
    #'rating_prom_id_libro',
    #'rating_prom_autor', 
    #'rating_prom_genero', 
    'n_autores_distintos_lector',
    'n_generos_distintos_lector'
]
features_dummies = [c for c in df_total.columns if c.startswith((
    "genero_persona_", 
    #"genero_libro_agrupado_", 
    #"editorial_agrupada_", 
    #"pais_persona_agrupado"
))]
features = features_base + features_dummies

X = df_total[features]
y = df_total["rating"]

In [55]:
#b. Split.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [56]:
#c. Busco hiperparametros con RandomizedSearchCV.
#i. Espacio de búsqueda.
#param_dist = {
#    "n_estimators": [100, 200, 500],
#    "max_depth": [10, 20, 30],
#    "min_samples_leaf": [3, 5, 10],
#    "min_samples_split": [5, 10],
#    "max_features": ["sqrt", "log2"],
#}

#ii. Armo la búsqueda.
#busqueda = RandomizedSearchCV(
#    estimator=RandomForestRegressor(random_state=42, n_jobs=1),
#    param_distributions=param_dist,
#    n_iter=30,
#    scoring="neg_root_mean_squared_error",
#    cv=3,
#    random_state=42,
#    verbose=2,
#    n_jobs=-1,
#)

#iii. Corro la búsqueda sobre train.
#busqueda.fit(X_train, y_train)

#iv. Reviso los mejores hiperparámetros encontrados.
#print("Mejores hiperparámetros:", busqueda.best_params_)
#print("Mejor RMSE (CV, promedio de los 3 folds):", -busqueda.best_score_)

# Si quiero dejar congelado los hiperparámetros.
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=30,
    min_samples_leaf=3,
    min_samples_split=10,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train,y_train)

,n_estimators,500
,criterion,'squared_error'
,max_depth,30
,min_samples_split,10
,min_samples_leaf,3
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [57]:
#d. Evaluo el mejor modelo encontrado, sobre el mismo test set de siempre (comparación justa contra el baseline: 1.68485).
#rf = busqueda.best_estimator_ # Descomentar si hice búsqueda. Comentar si están congelados los hiperparámetros.
preds = rf.predict(X_test)
print("RMSE en test con hiperparámetros optimizados:", root_mean_squared_error(y_test, preds))

RMSE en test con hiperparámetros optimizados: 1.4129004425309022


In [58]:
#e. Veo la importancia de features.
importancias = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(importancias.head(15))

rating_prom_id_lector_autor                   0.283
rating_prom_id_lector_genero_libro_agrupado   0.176
frecuencia_libro                              0.065
dias_transcurridos_interaccion                0.058
n_lectores_distintos_autor                    0.058
n_autores_distintos_lector                    0.048
frecuencia_lector                             0.048
n_interacciones_lector_genero                 0.042
edad_al_interactuar                           0.036
anios_transcurridos_edicion                   0.030
nacimiento                                    0.030
n_generos_distintos_lector                    0.029
n_interacciones_lector_autor                  0.029
antiguedad_libro_hoy                          0.026
anio_edicion                                  0.026
dtype: float64


In [41]:
#f. Reentreno el modelo FINAL, con los mejores hiperparámetros, sobre TODOS los datos (ya validé el desempeño arriba con el split).
rf.fit(X, y)

,n_estimators,500
,criterion,'squared_error'
,max_depth,30
,min_samples_split,10
,min_samples_leaf,3
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


#### 9. Resultado.

In [80]:
#a. Me quedo con los ids a predecir.
ids_lectores_entregar = df_entregar["id_lector"].unique()

In [81]:
#b. Armo tabla de atributos por lector (fijos, no dependen de la interacción puntual).
df_total["edad_actual_lector"] = anio_actual - df_total["nacimiento"]

atributos_lector = df_total.drop_duplicates("id_lector")[
    ["id_lector",
     "nacimiento",
     "edad_actual_lector",
     #"rating_prom_id_lector",
     "frecuencia_lector",
     "n_autores_distintos_lector",
     "n_generos_distintos_lector"]
    + [c for c in df_total.columns if c.startswith("genero_persona_")]
]

In [82]:
#c. Armo tabla de atributos por libro (fijos, UNA fila por libro), desde df_libros.
atributos_libro = df_libros[["id_libro", "autor", "genero_libro_agrupado", "anio_edicion"]].copy()
atributos_libro["antiguedad_libro_hoy"] = anio_actual - atributos_libro["anio_edicion"]

#i. Le sumo rating_prom_id_libro y frecuencia_libro (vienen de df_total, ya calculados, una fila por libro).
stats_libro_lookup = df_total.drop_duplicates("id_libro")[["id_libro", "frecuencia_libro"]] #"rating_prom_id_libro", 
atributos_libro = atributos_libro.merge(stats_libro_lookup, on="id_libro", how="left")

#ii. rating_prom_autor y rating_prom_genero: promedio PLENO (no LOO), sin target que filtrar en inferencia.
#stats_autor = df_total.groupby("autor")["rating"].mean().rename("rating_prom_autor")
#stats_genero = df_total.groupby("genero_libro_agrupado")["rating"].mean().rename("rating_prom_genero")
#atributos_libro = atributos_libro.merge(stats_autor, on="autor", how="left")
#atributos_libro = atributos_libro.merge(stats_genero, on="genero_libro_agrupado", how="left")

#iii. n_lectores_distintos_autor: popularidad PLENA del autor (sin el ajuste LOO que usamos en training).
stats_autor_popularidad = df_total.groupby("autor")["id_lector"].nunique().rename("n_lectores_distintos_autor")
atributos_libro = atributos_libro.merge(stats_autor_popularidad, on="autor", how="left")

In [83]:
#d. Tablas de afinidad lector x autor y lector x género.
afinidad_lector_autor = df_total[
    ["id_lector", "autor", "rating_prom_id_lector_autor", "n_interacciones_lector_autor"]
].drop_duplicates(subset=["id_lector", "autor"])

afinidad_lector_genero = df_total[
    ["id_lector", "genero_libro_agrupado", "rating_prom_id_lector_genero_libro_agrupado", "n_interacciones_lector_genero"]
].drop_duplicates(subset=["id_lector", "genero_libro_agrupado"])

In [84]:
#e. Libros ya leidos por cada lector (para no recomendar lo que ya leyo).
leidos_por_lector = df_total.groupby("id_lector")["id_libro"].apply(set)

In [85]:
#f. Todos los libros disponibles.
todos_los_libros = df_total["id_libro"].unique()

In [86]:
#g. Funcion que arma el top 20 para un lector.
#1. Defino variables claves.
cols_modelo = list(rf.feature_names_in_)  # fuente de verdad: exactamente lo que el modelo espera.
conteo_columnas_rellenadas = Counter()  # acumula, en todas las llamadas, qué columnas hubo que rellenar con 0 y cuántas veces.
media_global = y.mean()  # sin suavizado, uso la media global como fallback simple para combinaciones nunca vistas.
#2. Defino la función.
def top_20_random_forest(lector_id):
    #i. Libros ya leidos por este lector (set vacio si es nuevo, sin historial).
    leidos = leidos_por_lector.get(lector_id, set())

    #ii. Candidatos: todos los libros que este lector NO leyó todavía.
    candidatos = [libro for libro in todos_los_libros if libro not in leidos]
    if len(candidatos) == 0:
        return []

    #iii. Armo el dataframe base de candidatos y le pego los atributos del libro
    #     (autor, género, antigüedad, rating_prom_id_libro, rating_prom_autor,
    #     rating_prom_genero, n_lectores_distintos_autor, frecuencia_libro).
    df_cand = pd.DataFrame({"id_libro": candidatos})
    df_cand["id_lector"] = lector_id
    df_cand = df_cand.merge(atributos_libro, on="id_libro", how="left")

    #iv. Defensa extra: si atributos_libro viniera duplicado por id_libro, lo corrijo acá mismo.
    df_cand = df_cand.drop_duplicates(subset=["id_libro"])

    #v. Atributos del lector (constantes para todas las filas).
    lector_row = atributos_lector[atributos_lector["id_lector"] == lector_id]
    if lector_row.empty:  # Si es nuevo, entonces completo los datos de lector (uno solo).
        df_cand["nacimiento"] = df_total["nacimiento"].median()
        df_cand["edad_actual_lector"] = anio_actual - df_cand["nacimiento"]
        #df_cand["rating_prom_id_lector"] = media_global
        df_cand["frecuencia_lector"] = 0
        df_cand["n_autores_distintos_lector"] = 0
        df_cand["n_generos_distintos_lector"] = 0
        for col in atributos_lector.columns:
            if col.startswith("genero_persona_"):
                df_cand[col] = False
    else:
        for col in lector_row.columns:
            if col != "id_lector":
                df_cand[col] = lector_row[col].values[0]

    #vi. Afinidad lector x autor: rating promedio + cantidad de interacciones históricas.
    df_cand = df_cand.merge(afinidad_lector_autor, on=["id_lector", "autor"], how="left")
    df_cand["rating_prom_id_lector_autor"] = df_cand["rating_prom_id_lector_autor"].fillna(media_global)
    df_cand["n_interacciones_lector_autor"] = df_cand["n_interacciones_lector_autor"].fillna(0)

    #vii. Afinidad lector x género (idem).
    df_cand = df_cand.merge(afinidad_lector_genero, on=["id_lector", "genero_libro_agrupado"], how="left")
    df_cand["rating_prom_id_lector_genero_libro_agrupado"] = df_cand["rating_prom_id_lector_genero_libro_agrupado"].fillna(media_global)
    df_cand["n_interacciones_lector_genero"] = df_cand["n_interacciones_lector_genero"].fillna(0)

    #viii. Proxies para features que dependen de una interacción real que todavía no existe.
    df_cand["dias_transcurridos_interaccion"] = 0
    df_cand["edad_al_interactuar"] = df_cand["edad_actual_lector"]
    df_cand["anios_transcurridos_edicion"] = anio_actual - df_cand["anio_edicion"]

    #ix. Dummies de genero_libro_agrupado y editorial_agrupada (si algún día se activan como feature).
    #df_cand = pd.get_dummies(df_cand, columns=["genero_libro_agrupado", "editorial_agrupada"])

    #x. Relleno con 0 columnas que el modelo espera pero no se generaron.
    faltantes = [c for c in cols_modelo if c not in df_cand.columns]
    if faltantes:
        conteo_columnas_rellenadas.update(faltantes)
    for col in faltantes:
        df_cand[col] = 0

    #xi. Ordeno columnas EXACTAMENTE como el modelo las espera.
    X_cand = df_cand[cols_modelo]

    #xii. Predigo rating.
    df_cand["rating_predicho"] = rf.predict(X_cand)

    #xiii. Top 20 por rating predicho.
    top_20 = df_cand.sort_values("rating_predicho", ascending=False).head(20)
    top_20_ids = top_20["id_libro"].tolist()

    #xiv. Chequeo de integridad.
    assert len(top_20_ids) == len(set(top_20_ids)), f"IDs repetidos en el top 20 de {lector_id}"

    return top_20_ids

In [69]:
####################################################################################################################################
####################################################################################################################################
############################################################# PROBANDO #############################################################
####################################################################################################################################
####################################################################################################################################
#### Validación NDCG local, antes de exportar a Kaggle.
#### Corre después de entrenar `rf` (sección 8) y armar las tablas de la sección 9
#### (atributos_lector, atributos_libro, afinidad_lector_autor, afinidad_lector_genero, cols_modelo).

from sklearn.metrics import ndcg_score

def armar_candidatos_lector(lector_id, libros_a_evaluar):
    """Misma lógica que top_20_random_forest, pero para un set específico de libros (no excluye leídos)."""
    df_cand = pd.DataFrame({"id_libro": libros_a_evaluar})
    df_cand["id_lector"] = lector_id
    df_cand = df_cand.merge(atributos_libro, on="id_libro", how="left").drop_duplicates(subset=["id_libro"])

    lector_row = atributos_lector[atributos_lector["id_lector"] == lector_id]
    for col in lector_row.columns:
        if col != "id_lector":
            df_cand[col] = lector_row[col].values[0]

    df_cand = df_cand.merge(afinidad_lector_autor, on=["id_lector", "autor"], how="left")
    df_cand = df_cand.merge(afinidad_lector_genero, on=["id_lector", "genero_libro_agrupado"], how="left")
    for col in ["n_interacciones_lector_autor", "n_interacciones_lector_genero"]:
        if col in df_cand.columns:
            df_cand[col] = df_cand[col].fillna(0)

    df_cand["dias_transcurridos_interaccion"] = 0
    df_cand["edad_al_interactuar"] = df_cand["edad_actual_lector"]
    df_cand["anios_transcurridos_edicion"] = anio_actual - df_cand["anio_edicion"]

    faltantes = [c for c in cols_modelo if c not in df_cand.columns]
    for col in faltantes:
        df_cand[col] = 0

    X_cand = df_cand[cols_modelo]
    df_cand["rating_predicho"] = rf.predict(X_cand)
    return df_cand

lectores_holdout = (
    df_total.groupby("id_lector").size()
    .loc[lambda s: s >= 2]
    .sample(frac=0.1, random_state=42).index
)

scores_ndcg = []
for lector_id in lectores_holdout:
    reales = df_total[df_total["id_lector"] == lector_id][["id_libro", "rating"]].drop_duplicates("id_libro")
    df_cand_lector = armar_candidatos_lector(lector_id, reales["id_libro"].tolist())
    merged = reales.merge(df_cand_lector[["id_libro", "rating_predicho"]], on="id_libro", how="inner")
    if len(merged) < 2:
        continue
    score = ndcg_score([merged["rating"].values], [merged["rating_predicho"].values])
    scores_ndcg.append(score)

print(f"NDCG promedio en holdout: {sum(scores_ndcg)/len(scores_ndcg):.4f} (sobre {len(scores_ndcg)} lectores)")
####################################################################################################################################
####################################################################################################################################
####################################################################################################################################
####################################################################################################################################

NDCG promedio en holdout: 0.9499 (sobre 890 lectores)


In [87]:
#h. Genero recomendaciones para todos los lectores a entregar.
resultados = []
for lector_id in ids_lectores_entregar:
    top_20 = top_20_random_forest(lector_id)
    for libro in top_20:
        resultados.append({"id_lector": lector_id, "id_libro": libro})

df_resultado = pd.DataFrame(resultados)

In [88]:
#i. Reporto qué columnas tuvieron que rellenarse con 0, y cuántas veces (de un máximo de len(ids_lectores_entregar) lectores).
if conteo_columnas_rellenadas:
    print(f"⚠️  Se rellenaron columnas con 0 en algún momento (de {len(ids_lectores_entregar)} lectores procesados):")
    for col, veces in conteo_columnas_rellenadas.most_common():
        print(f"   - {col}: {veces} veces")
else:
    print("✅ Ninguna columna tuvo que rellenarse con 0 — el pipeline de candidatos coincide perfectamente con lo que el modelo espera.")

✅ Ninguna columna tuvo que rellenarse con 0 — el pipeline de candidatos coincide perfectamente con lo que el modelo espera.


In [89]:
#j. Chequeo que tenga el formato esperado.
print(df_resultado.shape)
df_resultado.head(25)

(16640, 2)


,id_lector,id_libro
0,05-03-1970,cien-anos-de-soledad
1,05-03-1970,carta-de-una-desconocida
2,05-03-1970,novela-de-ajedrez
3,05-03-1970,la-fiesta-del-chivo
4,05-03-1970,mendel-el-de-los-libros
5,05-03-1970,maria-antonieta
6,05-03-1970,el-amor-en-los-tiempos-del-colera-2
7,05-03-1970,americo-vespucio-relato-de-un-error-historico
8,05-03-1970,erasmo-de-rotterdam-triunfo-y-tragedia-de-un-h...
9,05-03-1970,americo-vespucio-la-historia-de-un-error-histo...


#### 10. Exportación.

In [90]:
#a. Defino el número de versión (a mano).
version = "10"

In [91]:
#b. Exporto la versión de esta corrida.
df_resultado.to_csv(f"./outputs/entregable_{version}.csv",index=False)